# CIFAR 10 Classification
## 1.2 Improving upon our existing baseline cnn archeticture

In [84]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import shutil

In [85]:
import numpy as np
import random
import os

seed = 42
# Setting Python and NumPy seeds
random.seed(seed)
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
    
# Setting PyTorch seeds (CPU and all GPUs)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
    
# Ensure deterministic behavior in cuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [86]:
# checking if CUDA is available and set the device accordingly,
# if a GPU is available it will use it for computations, otherwise it will fall back to using the CPU.
# in this case I am using RTX 3060 GPU which has CUDA support
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)

cuda:0


In [87]:
TRAIN_NETWORK_1 = True
TRAIN_NETWORK_2 = True

SAVE_TABLES = True
DELETE_TABLES = True

# if DELETE_TABLES is set to True, the code will check if the './tables' directory exists and if it does,
# it will iterate through all the files in that directory and delete them.
# If any file cannot be deleted, it will print an error message with the reason for the failure
# I used AI to make this deletion part so I don't know the details of how it works but
# I assume it uses the os and shutil modules to perform file operations
if DELETE_TABLES:
    if os.path.exists('./tables'):
        for filename in os.listdir('./tables'):
            file_path = os.path.join('./tables', filename)
            try:
                if os.path.isfile(file_path) or os.path.islink(file_path):
                    os.unlink(file_path)
                elif os.path.isdir(file_path):
                    shutil.rmtree(file_path)
            except Exception as e:
                print(f'Failed to delete {file_path}. Reason: {e}')

In [ ]:
# creating two empty pandas DataFrames with specified columns
# to store training times and accuracies for different models, for later analysis and comparisons
training_time_table = pd.DataFrame(columns=['Model', 'Training Time (seconds)'])
accuracy_table = pd.DataFrame(columns=['Model', 'Accuracy (%)'])
loss_table = pd.DataFrame(columns=['Model', 'Epoch', 'Loss'])

In [89]:
# same cell as previous notebook
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

# increased the batch size to 32 for faster gpu training
batch_size = 32

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=False, transform=transform)
trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=False, transform=transform)
testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [90]:
# same neural network architecture as previous notebook but with increased filters &
# neurons in the fully connected layers to see if it can achieve better performance

class Net (nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 5)
        
        self.pool= nn.MaxPool2d(2,2)
        
        self.conv2 = nn.Conv2d(16, 32, 5)
        
        self.fc1 = nn.Linear(32 * 5 * 5, 144)

        self.fc2 = nn.Linear(144, 98)

        self.fc3 = nn.Linear(98, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))

        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))

        x = F.relu(self.fc2(x))

        x = self.fc3(x)
        return x
    
net = Net()

net.to(device)

Net(
  (conv1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=800, out_features=144, bias=True)
  (fc2): Linear(in_features=144, out_features=98, bias=True)
  (fc3): Linear(in_features=98, out_features=10, bias=True)
)

In [91]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
if TRAIN_NETWORK_1:
    optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
if TRAIN_NETWORK_1:
    start = time.perf_counter()
    for epoch in range(10):
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            inputs, labels = data
            
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            outputs = net(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f'[Epoch {epoch + 1}] loss: {running_loss / len(trainloader):.3f}')
        loss_table.loc[len(loss_table)] = {'Model': 'Baseline CNN + Increased', 'Epoch': epoch + 1, 'Loss': running_loss / len(trainloader)}

    end = time.perf_counter()
    time_gpu_increased = end - start
    print(f'Training time on GPU with increased filters, neurons and batchsize: {time_gpu_increased:.2f} seconds')

[Epoch 1] loss: 2.127
[Epoch 2] loss: 1.678
[Epoch 3] loss: 1.470
[Epoch 4] loss: 1.341
[Epoch 5] loss: 1.237
[Epoch 6] loss: 1.147
[Epoch 7] loss: 1.072
[Epoch 8] loss: 1.007
[Epoch 9] loss: 0.952
[Epoch 10] loss: 0.903
Training time on GPU with increased filters, neurons and batchsize: 142.01 seconds


In [93]:
if not TRAIN_NETWORK_1:
    PATH = './models/cifar_net_gpu_increased.pth'
    net.load_state_dict(torch.load(PATH, map_location=device))

In [94]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_gpu_increased = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy_gpu_increased:.2f}%')

Accuracy of the network on the 10000 test images: 66.56%


In [ ]:
training_time_table.loc[len(training_time_table)] = {'Model': 'Baseline CNN + Increased', 'Training Time (seconds)': round(time_gpu_increased, 2)}
accuracy_table.loc[len(accuracy_table)] = {'Model': 'Baseline CNN + Increased', 'Accuracy (%)': round(accuracy_gpu_increased, 2)}

In [ ]:
if SAVE_TABLES:
    training_time_table.to_csv("tables/training_times.csv", index=False)
    accuracy_table.to_csv("tables/accuracy.csv", index=False)
    loss_table.to_csv("tables/loss.csv", index=False)

In [97]:
if TRAIN_NETWORK_1:
    PATH = './models/cifar_net_cpu_increased.pth'
    torch.save(net.state_dict(), PATH)

In [98]:
class Net2 (nn.Module):
    def __init__(self):
        super(Net2, self).__init__()
        # same first convolutional layer as previous network however I removed bias 
        # because of batch normalization layer will have offset trainable parameter anyway
        # kept padding the same to see if keeping the edge information of the input images 
        # will help the network to learn better features
        # the first convolutional layer will keep the same size of the original input images
        # which is 32x32 instead of reducing it to 28x28 as in the previous network
        self.conv1 = nn.Conv2d(3, 16 ,5 , padding="same", bias = False)
        # batch normalization layer after the first convolutional layer to normalize the activations
        # and improve training stability it takes the number of output channels
        # from the first convolutional layer which is 16 as an argument
        # z = (x - mean) / sqrt(var + eps) * gamma + beta(offset)
        # this will not change the size of the output from the first convolutional layer
        # which is 32x32
        self.bn1 = nn.BatchNorm2d(16)
        # max pooling as previous network to reduce the spatial dimensions of the feature maps by half
        # now the output size will be 16x16 instead of 14x14 as in the previous network
        self.pool= nn.MaxPool2d(2,2)
        # second convolutional layer similar to the previous network
        # but without bias also due to batch normalization layer after it
        self.conv2 = nn.Conv2d(16, 32, 5, bias = False)
        # batch normalization layer also taking the number of output channels
        # from the second convolutional layer which is 32
        self.bn2 = nn.BatchNorm2d(32)
        # same fully connected layers but with input size in first layer changed from 32*5*5 to 32*6*6 
        # due to the change in the output size of the second convolutional layer
        self.fc1 = nn.Linear(32 * 6 * 6, 144)
        self.fc2 = nn.Linear(144, 98)
        self.fc3 = nn.Linear(98, 10)
        
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
        
net2 = Net2()
net2.to(device)

Net2(
  (conv1): Conv2d(3, 16, kernel_size=(5, 5), stride=(1, 1), padding=same, bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(5, 5), stride=(1, 1), bias=False)
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=1152, out_features=144, bias=True)
  (fc2): Linear(in_features=144, out_features=98, bias=True)
  (fc3): Linear(in_features=98, out_features=10, bias=True)
)

In [99]:
# same optimizer as previous network
optimizer2 = optim.SGD(net2.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# same training loop as previous network but using net2 and optimizer2 to train 
# the new architecture with batch normalization and padding
if TRAIN_NETWORK_2:
    start = time.perf_counter()
    for epoch in range(10):
        running_loss = 0.0
        for data in trainloader:
            inputs, labels = data
            
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer2.zero_grad()
            
            outputs = net2(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer2.step()
            
            running_loss += loss.item()
        print(f'[Epoch {epoch + 1}] loss: {running_loss / len(trainloader):.3f}')
        
    end = time.perf_counter()
    time_gpu_increased_pd_bn = end - start
    print(f'Training time on GPU with increased filters, neurons, batchsize and batch normalization: {time_gpu_increased_pd_bn:.2f} seconds')

[Epoch 1] loss: 1.606
[Epoch 2] loss: 1.193
[Epoch 3] loss: 1.031
[Epoch 4] loss: 0.938
[Epoch 5] loss: 0.862
[Epoch 6] loss: 0.805
[Epoch 7] loss: 0.749
[Epoch 8] loss: 0.705
[Epoch 9] loss: 0.665
[Epoch 10] loss: 0.625
Training time on GPU with increased filters, neurons, batchsize and batch normalization: 149.15 seconds


In [101]:
PATH = './models/cifar_net_gpu_increased_pd_bn.pth'
torch.save(net2.state_dict(), PATH)

In [102]:
total = 0
correct = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = net2(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_gpu_increased_pd_bn = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy_gpu_increased_pd_bn:.2f}%')

Accuracy of the network on the 10000 test images: 72.85%


In [103]:
training_time_table.loc[len(training_time_table)] = {'Model': 'Baseline CNN + Increased + Padding + Batch Normalization', 'Training Time (seconds)': round(time_gpu_increased_pd_bn, 2)}
accuracy_table.loc[len(accuracy_table)] = {'Model': 'Baseline CNN + Increased + Padding + Batch Normalization', 'Accuracy (%)': round(accuracy_gpu_increased_pd_bn, 2)}

In [104]:
if SAVE_TABLES:
    training_time_table.to_csv("tables/training_times.csv", index=False)
    accuracy_table.to_csv("tables/accuracy.csv", index=False)